# Step 10 — Multi-expert BC Training

Trains one behavior cloning (BC) expert per municipality using `output/sim_trace_table.csv` (`sim_action` as label).
Saves models to `output/multi_expert_bc_models/`.

In [1]:
%pip install -q scikit-learn joblib pandas numpy


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path('../')
OUTPUT_DIR = ROOT / 'output'
TRACE_PATH = OUTPUT_DIR / 'sim_trace_table.csv'
ACTIONS_PATH = OUTPUT_DIR / 'valid_action_space.csv'
MODEL_DIR = OUTPUT_DIR / 'multi_expert_bc_models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not TRACE_PATH.exists():
    raise FileNotFoundError(f'Missing trace file: {TRACE_PATH}')

df = pd.read_csv(TRACE_PATH)
required_cols = [
    'municipality', 'activity', 'prev_activity', 'step_index',
    'time_since_case_start_hours', 'time_since_prev_hours',
    'branch_label', 'branch_confidence', 'rework_count_activity',
    'sim_action', 'case_id'
 ]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns in sim_trace_table.csv: {missing}')

feature_cols = [
    'municipality', 'activity', 'prev_activity', 'step_index',
    'time_since_case_start_hours', 'time_since_prev_hours',
    'branch_label', 'branch_confidence', 'rework_count_activity'
 ]
cat_cols = ['activity', 'prev_activity', 'branch_label']
num_cols = [c for c in feature_cols if c not in cat_cols]

train_df = df[feature_cols + ['sim_action', 'case_id']].copy()
train_df = train_df.dropna(subset=['sim_action']).reset_index(drop=True)

# Robust sim_action encoding: supports both integer ids and string action names.
sim_action_raw = train_df['sim_action'].astype(str).str.strip()
sim_action_num = pd.to_numeric(sim_action_raw, errors='coerce')

if sim_action_num.notna().all():
    train_df['sim_action'] = sim_action_num.astype(int)
else:
    if not ACTIONS_PATH.exists():
        raise FileNotFoundError(
            f'sim_action appears non-numeric, but action map is missing: {ACTIONS_PATH}'
        )
    actions_map_df = pd.read_csv(ACTIONS_PATH)
    needed = {'action_id', 'action_name'}
    if not needed.issubset(set(actions_map_df.columns)):
        raise ValueError(
            f'{ACTIONS_PATH} must contain columns: action_id and action_name'
        )

    action_name_to_id = {
        str(name).strip(): int(aid)
        for aid, name in zip(actions_map_df['action_id'], actions_map_df['action_name'])
    }
    mapped = sim_action_raw.map(action_name_to_id)
    if mapped.isna().any():
        unknown = sorted(sim_action_raw[mapped.isna()].unique().tolist())
        preview = unknown[:8]
        raise ValueError(
            f'Unmapped sim_action labels (showing up to 8): {preview}. '
            f'Please ensure names match {ACTIONS_PATH.name} action_name values.'
        )
    train_df['sim_action'] = mapped.astype(int)

print(
    f'Training rows: {len(train_df)} | municipalities: '
    f'{sorted(train_df["municipality"].unique().tolist())} | '
    f'actions: {sorted(train_df["sim_action"].unique().tolist())}'
)

Training rows: 405731 | municipalities: [1, 2, 3, 4, 5] | actions: [0, 2, 4, 6, 12, 14]


In [6]:
report_rows = []
metadata = {
    'feature_cols': feature_cols,
    'cat_cols': cat_cols,
    'num_cols': num_cols,
    'models': []
}

for municipality, g in train_df.groupby('municipality'):
    if len(g) < 200:
        print(f'Skipping municipality {municipality}: too few rows ({len(g)}).')
        continue

    case_ids = g['case_id'].dropna().astype(str).unique().tolist()
    if len(case_ids) >= 5:
        train_cases, test_cases = train_test_split(case_ids, test_size=0.2, random_state=42)
        tr = g[g['case_id'].astype(str).isin(train_cases)].copy()
        te = g[g['case_id'].astype(str).isin(test_cases)].copy()
    else:
        tr, te = train_test_split(g, test_size=0.2, random_state=42)

    pre = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
            ('num', 'passthrough', num_cols),
        ]
    )
    clf = RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced_subsample',
        min_samples_leaf=2,
    )
    pipe = Pipeline([('pre', pre), ('clf', clf)])

    pipe.fit(tr[feature_cols], tr['sim_action'])
    pred = pipe.predict(te[feature_cols])
    acc = float(accuracy_score(te['sim_action'], pred))

    model_path = MODEL_DIR / f'multi_expert_bc_M{int(municipality)}.joblib'
    joblib.dump(pipe, model_path)

    report_rows.append({
        'municipality': int(municipality),
        'train_rows': int(len(tr)),
        'test_rows': int(len(te)),
        'test_accuracy': acc,
        'model_path': str(model_path),
    })
    metadata['models'].append({
        'municipality': int(municipality),
        'path': str(model_path),
    })
    print(f'M{int(municipality)} | train={len(tr)} test={len(te)} acc={acc:.4f}')

report_df = pd.DataFrame(report_rows).sort_values('municipality').reset_index(drop=True)
report_path = OUTPUT_DIR / 'multi_expert_bc_training_report.csv'
meta_path = MODEL_DIR / 'multi_expert_bc_metadata.json'
report_df.to_csv(report_path, index=False)
meta_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print(f'Saved training report: {report_path}')
print(f'Saved metadata: {meta_path}')
display(report_df)

M1 | train=64770 test=15946 acc=0.9880
M2 | train=68803 test=17231 acc=0.9770
M3 | train=66250 test=16641 acc=0.9918
M4 | train=64217 test=16028 acc=0.9877
M5 | train=60659 test=15186 acc=0.9856
Saved training report: ../output/multi_expert_bc_training_report.csv
Saved metadata: ../output/multi_expert_bc_models/multi_expert_bc_metadata.json


,municipality,train_rows,test_rows,test_accuracy,model_path
0,1,64770,15946,0.987959,../output/multi_expert_bc_models/multi_expert_...
1,2,68803,17231,0.976960,../output/multi_expert_bc_models/multi_expert_...
2,3,66250,16641,0.991767,../output/multi_expert_bc_models/multi_expert_...
3,4,64217,16028,0.987709,../output/multi_expert_bc_models/multi_expert_...
4,5,60659,15186,0.985579,../output/multi_expert_bc_models/multi_expert_...
